# SniffTest Training Notebook

Self-contained Colab training notebook. No dependency on `training/train.py`.

**Flow:**
1. Install pinned deps (T4-tested)
2. Clone repo + launch env server
3. Load Qwen 1.5B + LoRA
4. SFT warm-start on expert trajectories
5. Smoke test the env pipeline
6. GRPO curriculum: easy → medium → hard
7. Save + plot

**Runtime:** T4 GPU (Runtime → Change runtime type). No HF token needed.


## 1 · Verify GPU

In [ ]:
!nvidia-smi

## 2 · Pinned Install (~5 min)

Pins: `vllm==0.9.2`, `triton==3.2.0`, `transformers==4.56.2`, `trl==0.22.2`


In [ ]:
import os

# Install all dependencies
# This cell MUST run first. It will trigger a kernel restart on Colab so that
# all packages (numpy, torch, etc.) are loaded from freshly installed versions
# rather than Colab pre-loaded stale copies.
# After the restart, re-run from Cell 2 onwards - do NOT re-run this cell.

!pip install --upgrade -qqq uv

if "COLAB_" not in "".join(os.environ.keys()):
    # Non-Colab: plain install
    !pip install unsloth trl datasets matplotlib
else:
    # Colab: install full stack without pinning numpy.
    # Kernel restart below ensures fresh C-extensions are loaded.
    !uv pip install -qqq --upgrade torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq --upgrade trl datasets matplotlib

!pip uninstall -y torchcodec 2>/dev/null || true
print("OK install done")

# Show installed stack
import importlib.metadata as _md
for pkg in ["unsloth", "vllm", "transformers", "trl", "torch", "numpy"]:
    try: print(f"  {pkg:14} = {_md.version(pkg)}")
    except Exception: print(f"  {pkg:14} = MISSING")

# Force kernel restart on Colab so updated numpy/torch C-extensions load cleanly.
# The session disconnects briefly - just continue from Cell 2 (do not rerun Cell 1).
if "COLAB_" in "".join(os.environ.keys()):
    print("\nRestarting kernel to load updated packages...")
    import os as _os; _os.kill(_os.getpid(), 9)


## 3 · Clone Repo

In [ ]:
%cd /content
!rm -rf /content/sniff-test
!git clone https://github.com/skrript/sniff-test.git /content/sniff-test
%cd /content/sniff-test
!pip install -q -e ".[train]"
print("OK repo ready")

## 4 · Verify Versions

In [ ]:
import importlib.metadata as _md
for pkg, want in [("vllm","0.9.2"),("triton","3.2.0"),
                  ("transformers","4.56.2"),("trl","0.22.2"),("unsloth","any")]:
    try: got = _md.version(pkg)
    except Exception: got = "MISSING"
    print(f"{pkg:14} = {got:12}  (want {want})")
# If wrong: !pip install --force-reinstall --no-deps vllm==0.9.2 triton==3.2.0 transformers==4.56.2 trl==0.22.2
# Then restart runtime and re-run from Cell 1.

## 5 · Launch Env Server

Starts the SniffTest uvicorn server as a background subprocess on port 8000.


In [ ]:
import subprocess, time, os, httpx

try:
    if "ENV_PROC" in globals() and ENV_PROC.poll() is None:
        ENV_PROC.terminate(); ENV_PROC.wait(timeout=5)
except Exception:
    pass

os.makedirs("/content/sniff-test/logs", exist_ok=True)
ENV_PROC = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "127.0.0.1", "--port", "8000",
     "--log-level", "warning"],
    cwd="/content/sniff-test",
)
for _ in range(60):
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=0.5).status_code == 200:
            print("OK env server up at :8000"); break
    except Exception:
        pass
    time.sleep(0.5)
else:
    raise RuntimeError("Server didn't start — check uvicorn logs")

## 6 · Imports, Config, Load Model + LoRA

In [ ]:
import os, sys, json, collections, gc, torch
sys.path.insert(0, "/content/sniff-test")
os.environ["SNIFFTEST_ENV_URL"] = "http://127.0.0.1:8000"

from unsloth import FastLanguageModel
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

MODEL_NAME         = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_COMPLETION_LEN = 512
NUM_GENERATIONS    = 2
PER_DEVICE_BATCH   = 1
GRAD_ACCUM         = 2
LR                 = 5e-6
SMOKE_STEPS        = 10
RUN_ID             = "snifftest_run_a"
CKPT_ROOT          = f"/content/sniff-test/checkpoints/{RUN_ID}"
os.makedirs(CKPT_ROOT, exist_ok=True)

# ── Monkey-patch: vLLM 0.11.x calls all_special_tokens_extended on slow
# tokenizers (e.g. Qwen2Tokenizer) which don't implement it. Patch the base
# class so all tokenizer subclasses get a safe fallback.
try:
    from transformers import tokenization_utils_base as _tub
    if not hasattr(_tub.PreTrainedTokenizerBase, "all_special_tokens_extended"):
        _tub.PreTrainedTokenizerBase.all_special_tokens_extended = property(
            lambda self: list(self.all_special_tokens)
        )
        print("OK patched all_special_tokens_extended on PreTrainedTokenizerBase")
except Exception as _e:
    print(f"Warning: tokenizer patch failed ({_e}) — may still work")

SYSTEM_PROMPT = (
    "You are an expert fact-checker investigating claims for accuracy.\n\n"
    "Available actions (return as JSON, one action per response):\n"
    '{"action_type": "search", "query": "your search query"}\n'
    '{"action_type": "open_source", "source_id": "src_xxx"}\n'
    '{"action_type": "cross_reference", "source_ids": ["src_xxx", "src_yyy"]}\n'
    '{"action_type": "trace_origin", "source_id": "src_xxx"}\n'
    '{"action_type": "check_metadata", "source_id": "src_xxx"}\n'
    '{"action_type": "submit_verdict", "verdict": "true|false|misleading|unverifiable", '
    '"confidence": 0.0, "justification": "cite source IDs"}\n\n'
    "Strategy: search → open key sources → cross-reference or check metadata → submit verdict.\n"
    "Return ONLY the JSON action. No other text."
)

# ── Load model: try with vLLM (fast_inference=True), fall back to HF generate.
USE_VLLM = True
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name             = MODEL_NAME,
        max_seq_length         = MAX_COMPLETION_LEN + 2048,
        load_in_4bit           = True,
        fast_inference         = True,
        gpu_memory_utilization = 0.5,
        max_lora_rank          = 16,
        enforce_eager          = True,
    )
    print("OK loaded with fast_inference=True (vLLM colocate)")
except Exception as _vllm_err:
    print(f"vLLM init failed: {_vllm_err}")
    print("Falling back to fast_inference=False (standard HF generate — slower but safe)")
    USE_VLLM = False
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name    = MODEL_NAME,
        max_seq_length = MAX_COMPLETION_LEN + 2048,
        load_in_4bit  = True,
        fast_inference = False,
    )

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32, lora_dropout=0.0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print(f"OK model + LoRA ready | USE_VLLM={USE_VLLM} | TRL={GRPOTrainer.__module__}")
print(f"   Tokenizer type: {type(tokenizer).__name__}")


## 7 · SFT Warm-Start (optional but recommended)

Trains on expert trajectories from `data/sft_trajectories.jsonl` before RL.


In [ ]:
from trl import SFTConfig, SFTTrainer
import json, pathlib

SFT_DIR = f"{CKPT_ROOT}/sft"
TRAJ_PATH = "/content/sniff-test/data/sft_trajectories.jsonl"

def _load_sft_examples(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def _build_sft_text(record):
    visible = record.get("visible_sources", [])
    src_lines = "\n".join(
        f"- [{s['source_id']}] {s['title']} ({s['domain']}): {s['snippet']}"
        for s in visible
    )
    actions = record.get("actions", [])
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Investigate: {record['claim']}\n\nSources:\n{src_lines}\n\nReturn the next best single JSON action."},
        {"role": "assistant", "content": json.dumps(actions[-1], separators=(",",":"))},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

examples = _load_sft_examples(TRAJ_PATH)
texts = [{"text": _build_sft_text(r)} for r in examples if len(r.get("visible_sources",[])) >= 3]
sft_ds = Dataset.from_list(texts)
print(f"SFT examples: {len(texts)}")

sft_trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=sft_ds,
    args=SFTConfig(
        output_dir=SFT_DIR, num_train_epochs=2,
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        learning_rate=2e-4, bf16=False, fp16=True,
        max_seq_length=MAX_COMPLETION_LEN+2048,
        dataset_text_field="text", logging_steps=10, report_to="none",
    ),
)
sft_trainer.train()
model.save_pretrained(SFT_DIR); tokenizer.save_pretrained(SFT_DIR)
print(f"SFT saved -> {SFT_DIR}")

## 8 · SniffTestEnvFactory

Wraps the live env server via `SniffTestEnv` (OpenEnv WebSocket client).
TRL introspects the public tool methods and drives the multi-turn loop.

> **Why `environment_factory` and not `rollout_func`?**  
> Unsloth silently ignores `rollout_func` (GitHub issue #3573) and runs its
> own internal generation path, giving reward=0 every step.


In [ ]:
from client import SniffTestEnv
from models import InvestigateAction

ENV_URL = os.environ.get('SNIFFTEST_ENV_URL', 'http://127.0.0.1:8000')
MAX_STEPS = 10
ROLLOUT_METRICS_Q = collections.deque(maxlen=4096)

def _obs_to_text(obs, step):
    lines = [
        f'Step {step}/{MAX_STEPS}  |  Remaining: {obs.steps_remaining}',
        f'\nCLAIM: {obs.claim}',
        f'\nSOURCES ({len(obs.available_sources)}):',
    ]
    for s in obs.available_sources:
        tag = ' [READ]' if s.retrieved else ''
        lines.append(f'  [{s.source_id}] {s.title} ({s.domain}){tag}\n    {s.snippet}')
    if obs.opened_content: lines += ['\nOPENED:', obs.opened_content[:500]]
    if obs.cross_reference_result: lines += ['\nCROSS-REF:', obs.cross_reference_result[:300]]
    if obs.trace_result: lines += ['\nTRACE:', obs.trace_result[:300]]
    if obs.metadata_result: lines += ['\nMETADATA:', obs.metadata_result[:300]]
    if obs.action_history:
        lines.append('\nHISTORY (last 5):')
        for h in obs.action_history[-5:]:
            lines.append(f'  [{h.step}] {h.action_type}: {h.result_summary}')
    if obs.message: lines.append(f'\nFEEDBACK: {obs.message}')
    lines.append('\nNext action (JSON only):')
    return '\n'.join(lines)


class SniffTestEnvFactory:
    # TRL environment_factory: one instance per rollout, connects via WebSocket.
    # Use environment_factory=SniffTestEnvFactory in GRPOTrainer.
    # Unsloth ignores rollout_func (issue #3573); this is the correct API.

    def __init__(self):
        self._client = SniffTestEnv(base_url=ENV_URL).sync()
        self._client.__enter__()
        self.reward = 0.0
        self.done   = False
        self._step  = 0
        self._obs   = None
        self._closed = False

    def reset(self, task_level: str = 'easy', **kwargs) -> str:
        # Start a new investigation episode.
        result = self._client.reset(task_level=task_level)
        self.reward = 0.0; self.done = False; self._step = 0
        self._obs = result.observation
        return _obs_to_text(self._obs, step=1)

    def close(self):
        if self._closed: return
        self._closed = True
        if not self.done:
            try:
                r = self._client.step(InvestigateAction(
                    action_type='submit_verdict', verdict='unverifiable',
                    confidence=0.1, justification='auto-close'))
                self._on_terminal(r)
            except Exception: pass
        try: self._client.__exit__(None, None, None)
        except Exception: pass

    def __del__(self):
        try: self.close()
        except Exception: pass

    def _act(self, action):
        if self.done: return 'Episode ended.'
        self._step += 1
        result = self._client.step(action)
        self._obs = result.observation
        if result.done: self._on_terminal(result)
        return _obs_to_text(self._obs, self._step + 1)

    def _on_terminal(self, result):
        self.reward = float(result.reward or 0.0)
        self.done = True
        ROLLOUT_METRICS_Q.append({'reward': self.reward,
            'components': getattr(result.observation, 'reward_components', None) or {}})

    def search(self, query: str) -> str:
        # Search for sources about the claim. Args: query (str). Returns source list.
        return self._act(InvestigateAction(action_type='search', query=query))

    def open_source(self, source_id: str) -> str:
        # Read full content of a source. Args: source_id e.g. 'src_001'. Returns article text.
        return self._act(InvestigateAction(action_type='open_source', source_id=source_id))

    def cross_reference(self, source_id_a: str, source_id_b: str) -> str:
        # Compare two sources for contradictions. Returns AGREE/CONTRADICT analysis.
        return self._act(InvestigateAction(action_type='cross_reference', source_ids=[source_id_a, source_id_b]))

    def trace_origin(self, source_id: str) -> str:
        # Trace how a source propagated. Returns propagation chain.
        return self._act(InvestigateAction(action_type='trace_origin', source_id=source_id))

    def check_metadata(self, source_id: str) -> str:
        # Inspect credibility signals for a source. Returns metadata report.
        return self._act(InvestigateAction(action_type='check_metadata', source_id=source_id))

    def submit_verdict(self, verdict: str, confidence: float, justification: str) -> str:
        # Submit verdict (true|false|misleading|unverifiable), confidence 0-1, justification citing sources.
        return self._act(InvestigateAction(
            action_type='submit_verdict', verdict=verdict,
            confidence=float(confidence), justification=justification))

print('OK SniffTestEnvFactory defined')

## 9 · Pre-Train Smoke Test (~5 sec)

Catches broken env or tool dispatch before wasting GPU time.


In [ ]:
import httpx as _hx
# 1. Server health
r = _hx.get("http://127.0.0.1:8000/health", timeout=3.0)
assert r.status_code == 200, f"Server not reachable: {r.status_code} — re-run Cell 5"
print("OK server reachable")

# 2. One full rollout through the factory
env = SniffTestEnvFactory()
text = env.reset(task_level="easy")
print(f"OK reset -> {len(text)} chars")
r1 = env.search("fact check evidence")
print(f"OK search -> {r1[:80]!r}")
src = env._obs.available_sources[0].source_id if env._obs.available_sources else "src_001"
env.open_source(src)
print(f"OK open_source {src}")
env.submit_verdict("false", 0.4, f"Smoke test. Opened {src}.")
print(f"OK submit_verdict | reward={env.reward:.4f} | done={env.done}")
env.close()

assert env.done, "done flag not set after submit_verdict"
assert len(ROLLOUT_METRICS_Q) > 0, "Queue empty — _on_terminal never fired"
ROLLOUT_METRICS_Q.clear()
print("\nOK smoke passed — proceed to Cell 10")

## 10 · Reward Function + GRPOConfig

In [ ]:
def reward_func(completions, environments=None, **kwargs):
    # Read terminal reward from each env instance after rollout ends.
    if environments:
        return [float(getattr(e, "reward", 0.0)) for e in environments]
    return [0.0] * len(completions)

def _make_dataset(task_level, episodes):
    return Dataset.from_dict({"prompt": [
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": task_level}]
    ] * episodes})

def _make_grpo_config(output_dir, steps):
    # Build GRPOConfig that works with both vLLM and non-vLLM paths.
    # USE_VLLM is set in the model loading cell.
    kwargs = dict(
        output_dir=output_dir,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_generations=NUM_GENERATIONS,
        learning_rate=LR, lr_scheduler_type="cosine",
        max_completion_length=MAX_COMPLETION_LEN,
        max_steps=steps, logging_steps=1,
        save_steps=50, save_total_limit=2,
        log_completions=True, report_to="none",
        fp16=True, bf16=False, seed=42,
    )
    if USE_VLLM:
        kwargs["use_vllm"] = True
        # vllm_mode="colocate" only exists in certain TRL versions — try it.
        import inspect
        sig = inspect.signature(type("GRPOConfig", (), {}).__init__)
        try:
            cfg = GRPOConfig(**kwargs, vllm_mode="colocate")
            return cfg
        except TypeError:
            pass  # vllm_mode not supported in this TRL version, skip it
    return GRPOConfig(**kwargs)

print(f"OK reward_func + helpers ready | USE_VLLM={USE_VLLM}")


## 11 · Smoke GRPO (10 steps) — inspect reward before full run

In [ ]:
smoke_trainer = GRPOTrainer(
    model=model, processing_class=tokenizer,
    reward_funcs=reward_func,
    train_dataset=_make_dataset("easy", 64),
    args=_make_grpo_config(f"{CKPT_ROOT}/smoke", SMOKE_STEPS),
    environment_factory=SniffTestEnvFactory,
)
smoke_trainer.train()

rewards = [e["reward"] for e in smoke_trainer.state.log_history if "reward" in e]
print(f"Smoke rewards: {rewards}")
print("Non-zero -> pipeline OK. All zeros -> debug Cell 9.")

## 12 · Full Curriculum: easy → medium → hard

In [ ]:
CURRICULUM = [
    {"task_level": "easy",   "episodes": 200},
    {"task_level": "medium", "episodes": 200},
    {"task_level": "hard",   "episodes": 100},
]
all_rewards = []

trainer = GRPOTrainer(
    model=model, processing_class=tokenizer,
    reward_funcs=reward_func,
    train_dataset=_make_dataset("easy", 200),
    args=_make_grpo_config(f"{CKPT_ROOT}/easy", 20),
    environment_factory=SniffTestEnvFactory,
)

for stage in CURRICULUM:
    level, episodes = stage["task_level"], stage["episodes"]
    steps = max(10, episodes // 10)
    out   = f"{CKPT_ROOT}/{level}"
    print(f"\n{'='*50}\nSTAGE: {level.upper()} | {episodes} eps | {steps} steps\n{'='*50}")
    trainer.args          = _make_grpo_config(out, steps)
    trainer.train_dataset = _make_dataset(level, episodes)
    trainer.train(resume_from_checkpoint=False)
    for e in trainer.state.log_history:
        if "reward" in e:
            all_rewards.append({"level": level, "reward": e["reward"], "step": e.get("step")})
    recent = [r["reward"] for r in all_rewards if r["level"] == level][-10:]
    print(f"  recent avg reward: {sum(recent)/len(recent):.4f}" if recent else "  no rewards logged")
    model.save_pretrained(out); tokenizer.save_pretrained(out)
    print(f"  checkpoint: {out}")
    gc.collect(); torch.cuda.empty_cache()

import pathlib
hist_path = pathlib.Path(f"{CKPT_ROOT}/reward_history.json")
hist_path.write_text(json.dumps(all_rewards, indent=2))
print(f"\nReward history -> {hist_path}")

## 13 · Save Final Model + Reward Curve

In [ ]:
# Save merged 16-bit model for inference
final = f"{CKPT_ROOT}/final_merged"
model.save_pretrained_merged(final, tokenizer, save_method="merged_16bit")
print(f"Merged model -> {final}")

# Plot
import pandas as pd, matplotlib.pyplot as plt
df = pd.DataFrame(all_rewards)
if not df.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    for lvl, grp in df.groupby("level"):
        ax.plot(range(len(grp)), grp["reward"].values, label=lvl, alpha=0.8)
    ax.axhline(0, ls="--", color="grey", alpha=0.4)
    ax.set_xlabel("Training step"); ax.set_ylabel("Mean episode reward")
    ax.set_title("SniffTest GRPO — Qwen2.5-1.5B + Unsloth")
    ax.legend(); fig.tight_layout()
    png = f"{CKPT_ROOT}/reward_curve.png"
    fig.savefig(png, dpi=150); plt.show()
    print(f"Curve -> {png}")